# Session 6 — Measuring Agent Performance

**Spine:** *Comparing two versions is flipping two coins: report the interval, not the winner — and when it crosses zero, the tie goes to the cheaper agent.*

The deck carries the argument; this notebook carries only what you run. Every hands-on is a script in the repo — nothing is hidden, you just will not type it. The one file you **edit** is `criteria6.py`.

**Nothing today calls an agent.** The instructor ran 3 versions × 12 rows × 5 repetitions last night (180 runs). You work from `runs6.json`. Tavily cost today: **zero**. Only Hands-on 2 and 4 touch LangSmith, and every block after them works without it.

In [ ]:
!git pull

In [ ]:
# Environment check. Prints WHERE you are before importing anything, so a wrong
# kernel gives a named error instead of a spinner.
import sys, os
print("python:", sys.executable)
print("cwd   :", os.getcwd())
assert os.path.exists("runs6.json"), "runs6.json missing: are you in course-repo? did git pull work?"

from dotenv import load_dotenv
load_dotenv()          # every LangSmith call needs this. push_pool.py skipped it once; nothing got pushed.
print("LangSmith key loaded:", bool(os.environ.get("LANGSMITH_API_KEY")))

import importlib, json, statistics
import paired as P
import consistency6
runs = P.load("runs6.json")
print(f"runs6.json: {len(runs)} runs,", sorted({r['version'] for r in runs}))

## Opener — where does the noise live? *(projector)*

Session 5: *σ/µ ≈ 40%, so a 10% claim costs 252 runs per arm.* That σ came from **one** question, run four times.

Last night's healthy agent: **11 questions, 5 runs each, nothing changed between runs.** Before you look — which kind of question do you think moves the most?

In [ ]:
# PROJECTOR. healthy, tokens_billed, each question's five runs. Sorted quietest first.
rows = P.by_row(runs, "healthy", "tokens_billed")
table = []
for q, v in rows.items():
    m, sd = statistics.fmean(v), statistics.stdev(v)
    table.append((sd / m, q, min(v), max(v), m))
print(f"{'CV':>5}  {'lowest':>7} {'highest':>8} {'mean':>7}   question")
for cv, q, lo, hi, m in sorted(table):
    print(f"{cv:5.2f}  {lo:7.0f} {hi:8.0f} {m:7.0f}   {q[:62]}")

In [ ]:
# PROJECTOR. The same data as three numbers, and what each costs.
# n per arm for a 10% claim, 95% confidence, 80% power: 2 * (1.96 + 0.84)^2 * CV^2 / 0.10^2
n_for = lambda cv: round(2 * (1.96 + 0.8416) ** 2 * cv ** 2 / 0.01)
cvs = sorted(t[0] for t in table)
s_all, df, m_all = P.within_row_sigma(runs, "healthy", "tokens_billed")
share = P.variance_share(runs, "healthy", "tokens_billed")[0]
print(f"median question   CV {statistics.median(cvs):.2f}  -> {n_for(statistics.median(cvs)):>4} runs per arm for 10%")
print(f"noisiest question CV {cvs[-1]:.2f}  -> {n_for(cvs[-1]):>4} runs per arm for 10%")
print(f"pooled, all rows  CV {s_all / m_all:.2f}  -> {n_for(s_all / m_all):>4}   <- belongs to no question")
print(f"\n{share[1]:.0%} of all the run-to-run variance is one question:\n  {share[0]}")

## Hands-on 1 — your criteria, before any number *(8 min)*

Edit **`criteria6.py`** in VS Code. Three things:

1. **`SUCCESS_BAR`** — the pass rate an agent must reach to be production-ready at all.
2. **`ACT_IF`** — the smallest change you would *act on*. Not the smallest you could detect.
3. **`PREDICT`** — for `redundant` (*"run every search twice"*) and `concise` (*"Answer in 2 sentences."*): HIGHER / LOWER / TIE on each metric, versus healthy.

Then run the checker until it says OK. **Your predictions are locked when it does** — Hands-on 5 scores them.

*A threshold chosen after you have seen the result is not a threshold, it is a caption.*

In [ ]:
!python criteria6.py

## Hands-on 2 — the runs, in YOUR LangSmith *(10 min)*

```bash
python replay6.py --dry     # key + file check, touches nothing
python replay6.py           # pushes the v1 pool if you lack it, then 3 experiments
```

A **replay**: a target that returns a saved output instead of calling a model. Zero model cost, zero Tavily, three real experiments in your workspace.

**⚠ The trap.** LangSmith will show **~0.0 s latency and 0 tokens** for every replayed run — nothing was called. Read the **feedback** columns (`tokens_billed`, `n_searches`, `latency_s`), which carry what the agent recorded when it really ran.

Then in the UI: Datasets → `s5-class-benchmark-pool` → Experiments → tick two → **Compare**.

*No key, or a 401?* Skip it. Every block after this runs from `runs6.json` alone.

In [ ]:
!python replay6.py --dry

In [ ]:
!python replay6.py

## Hands-on 3 — the same question, five times *(8 min)*

```bash
python consistency6.py
```

**pass@1** is how good the agent is on average. **pass^5** is whether you can rely on it: a question that passes 4 runs in 5 has pass@1 = 0.8 and pass^5 = 0.

Answer with your partner, each with a number:
1. Which metric has the biggest gap between pass@1 and pass^5?
2. Session 5 said σ/µ ≈ 40%. For which questions is that true?

In [ ]:
!python consistency6.py

## Hands-on 4 — a benchmark becomes a regression set *(6 min · sacrificial)*

```bash
python make_regression_set.py --dry
python make_regression_set.py
```

A benchmark row says what a **right** answer looks like. A regression row also says what the **incumbent did** on it — median tokens and searches over healthy's five runs. The next candidate is scored against that row, by a rule: *within 25% of what production does today, on the same question.*

In [ ]:
!python make_regression_set.py --dry

In [ ]:
!python make_regression_set.py

## Hands-on 5 — the regression test *(12 min)* ⚑ THE SESSION

```bash
python regress6.py
```

For each candidate, each metric: the change, its **95% interval**, and what it means against **your** `ACT_IF` — TIE · REAL, BELOW BAR · REAL, MAYBE BAR · CLEARS BAR. Then your predictions, scored. Then the decision, by the tie rule.

In [ ]:
!python regress6.py

In [ ]:
# PROJECTOR. concise vs healthy, tokens, question by question.
pcts = P.row_pcts(runs, "healthy", "concise", "tokens_billed")
for q, v in sorted(pcts.items(), key=lambda kv: kv[1]):
    print(f"{v:+6.0f}%   {q[:70]}")
print(f"\nmean of the paired difference: {P.paired(runs, 'healthy', 'concise', 'tokens_billed').pct:+.0f}%")
print(f"median question:               {statistics.median(pcts.values()):+.0f}%")

## Automating it *(instructor demo · sacrificial)*

A regression test someone has to remember to run stops being run. `regress_gate.py` is the same arithmetic reduced to what CI understands: an **exit code**.

In [ ]:
!python regress_gate.py --cand redundant; echo "exit code: $?"
!python regress_gate.py --cand concise;   echo "exit code: $?"

In [ ]:
# Why the gate pairs runs itself instead of using LangSmith's evaluate_comparative.
# Measured by preflight6, 10 Sep: with num_repetitions=5 the comparator received
# this many runs per example. The docs describe a two-item list.
print(json.load(open("deck_numbers6.json")).get("comparative_runs_per_example"))

## Hands-on 6 — the report, and your recommendation *(8 min)*

```bash
python report6.py
```

It writes `report6_<you>.md`. Everything is generated **except section 5**: one sentence, by hand, and it **must quote an interval**.

- Not allowed: *"concise is 26% cheaper."*
- Allowed: *"concise's token change is −26%, 95% interval −75% to +23%, so we keep healthy."*

This is the shape of the mid-term.

In [ ]:
!python report6.py